## Structured out put

In [17]:
!pip install groq
!pip install pydantic
!pip instal openai

ERROR: unknown command "instal" - maybe you meant "install"


In [18]:
from IPython.core.display import json
from groq import Groq
from pydantic import BaseModel
from typing import Literal
import json
from openai import OpenAI

In [30]:
from httpx import Response
client = OpenAI(base_url="https://openrouter.ai/api/v1",
    api_key="Key")

class productReview(BaseModel):
    product_name: str
    rating: float
    sentiment: Literal["Positive","Negative","neutral"]

# Use beta.chat.completions.parse for automatic Pydantic validation

completion = client.beta.chat.completions.parse(
    model="mistralai/mistral-medium-3-5", # Note: Ensure the model supports structured outputs/JSON mode
    messages=[
        {"role": "system", "content": "Extract the product review information from the text."},
        {"role": "user", "content": "I bought the UltraSound Headphones last week and I'm really impressed! The noise cancellation is amazing and the battery lasts all day. Sound quality is crisp and clear. I'd give it 4.5 out of 5 stars."}
    ],
     response_format=productReview,
)

# The parsed Pydantic object is ready immediately!
review = completion.choices[0].message.parsed
print(review)
print(review.product_name)

product_name='UltraSound Headphones' rating=4.5 sentiment='Positive'
UltraSound Headphones


In [28]:
review

productReview(product_name='UltraSound Headphones', rating=4.5, sentiment='Positive')

In [29]:
class KeyEntity(BaseModel):
  entity: str
  type: str

class EmailClassification(BaseModel):
  category: str
  priority: str
  confidance_level: float
  sentiment: str
  key_entities: list[KeyEntity]
  suggested_actions: list[str]
  requires_immediate_attention: bool
  estimated_response_time: str



In [35]:
completion = client.beta.chat.completions.parse(
    model="mistralai/mistral-medium-3-5",
    temperature= 0.0,
    messages=[
        {"role":"system","content":"you are an expert email classifier who classifies emails into structure catagories with confidance scores, priority level,and suggested action."},
        {"role":"user","content":"Subject: URGENT: Server downtime affecting production\\n\\nHi Team,\\n\\nOur main production server went down at 2:30 PM EST. Customer-facing services are currently unavailable. We need immediate action to restore services. Please join the emergency call.\\n\\nBest regards,\\nDevOps Team"}
    ],
    response_format=EmailClassification
)
email = completion.choices[0].message.parsed
print(email)

category='Technical Incident' priority='Critical' confidance_level=0.98 sentiment='Negative' key_entities=[KeyEntity(entity='main production server', type='Infrastructure'), KeyEntity(entity='Customer-facing services', type='Service'), KeyEntity(entity='DevOps Team', type='Team')] suggested_actions=['Initiate emergency response protocol', 'Join the emergency call immediately', 'Investigate root cause of server downtime', 'Restore services as soon as possible', 'Communicate updates to stakeholders'] requires_immediate_attention=True estimated_response_time='Immediate (within 15 minutes)'


In [41]:
from typing import List, Optional, Literal
from enum import Enum
import json
from pydantic import Field

In [42]:
class SupportCategory(str, Enum):
    API = "api"
    BILLING = "billing"
    ACCOUNT = "account"
    BUG = "bug"
    FEATURE_REQUEST = "feature_request"
    INTEGRATION = "integration"
    SECURITY = "security"
    PERFORMANCE = "performance"

class Priority(str, Enum):
    LOW = "low"
    MEDIUM = "medium"
    HIGH = "high"
    CRITICAL = "critical"

class CustomerTier(str, Enum):
    FREE = "free"
    PAID = "paid"
    ENTERPRISE = "enterprise"
    TRIAL = "trial"

class CustomerInfo(BaseModel):
    name: str
    company: Optional[str] = None
    tier: CustomerTier

class TechnicalDetail(BaseModel):
    component: str
    error_code: Optional[str] = None
    description: str

class SupportTicket(BaseModel):
    category: SupportCategory
    priority: Priority
    urgency_score: float
    customer_info: CustomerInfo
    technical_details: List[TechnicalDetail]
    keywords: List[str]
    requires_escalation: bool
    estimated_resolution_hours: float
    follow_up_date: Optional[str] = Field(None, description="ISO datetime string")
    summary: str

In [47]:
completion = client.beta.chat.completions.parse(
    model="mistralai/mistral-medium-3-5",
    temperature= 1.0,
    messages=[
        {"role":"system","content":"""You are a customer support ticket classifier for SaaS companies.
                         Analyze support tickets and categorize them for efficient routing and resolution.
                         Output JSON only using the schema provided."""},
        {"role":"user","content":"""Hello! I love your product and have been using it for 6 months.
                         I was wondering if you could add a dark mode feature to the dashboard?
                         Many of our team members work late hours and would really appreciate this.
                         Also, it would be great to have keyboard shortcuts for common actions.
                         Not urgent, but would be a nice enhancement!
                         Best, Mike from StartupXYZ"""}
    ],
    response_format=SupportTicket
)
result = completion.choices[0].message.parsed
print(result)

category=<SupportCategory.FEATURE_REQUEST: 'feature_request'> priority=<Priority.LOW: 'low'> urgency_score=1.5 customer_info=CustomerInfo(name='Mike', company='StartupXYZ', tier=<CustomerTier.FREE: 'free'>) technical_details=[TechnicalDetail(component='dashboard', error_code=None, description='Request for dark mode feature and keyboard shortcuts for common actions')] keywords=['dark mode', 'dashboard', 'keyboard shortcuts', 'enhancement', 'UI improvement'] requires_escalation=False estimated_resolution_hours=40.0 follow_up_date=None summary='Feature request for dark mode and keyboard shortcuts in the dashboard to improve user experience for late-night workers.'


In [48]:
print(result.model_dump_json(indent=2))

{
  "category": "feature_request",
  "priority": "low",
  "urgency_score": 1.5,
  "customer_info": {
    "name": "Mike",
    "company": "StartupXYZ",
    "tier": "free"
  },
  "technical_details": [
    {
      "component": "dashboard",
      "error_code": null,
      "description": "Request for dark mode feature and keyboard shortcuts for common actions"
    }
  ],
  "keywords": [
    "dark mode",
    "dashboard",
    "keyboard shortcuts",
    "enhancement",
    "UI improvement"
  ],
  "requires_escalation": false,
  "estimated_resolution_hours": 40.0,
  "follow_up_date": null,
  "summary": "Feature request for dark mode and keyboard shortcuts in the dashboard to improve user experience for late-night workers."
}
